# Atewa Range Forest Reserve — Deforestation Monitoring Template

**Built for the Forestry Commission of Ghana — designed to be reused directly for any other reserve, not just Atewa.**

Atewa Range Forest Reserve (Eastern Region, ~90km north of Accra) is one
of only two Upland Evergreen forest sites in Ghana, source of the
Densu, Birim, and Ayensu rivers, and the subject of a major ongoing
national debate over a proposed bauxite mine within its boundaries. It
is exactly the kind of site where an independent, repeatable, real-data
monitoring pipeline matters.

**Optical AND radar, deliberately.** Atewa's upland evergreen forest
sits in one of Ghana's persistently cloudy zones — optical-only
monitoring in tropical forest is a known, real limitation, not a
hypothetical one. This notebook combines a Landsat NDVI trend with a
Sentinel-1 SAR (VH backscatter) cross-check for exactly that reason;
see Section 11 for the full rationale.

**No synthetic data anywhere.** Every result reflects live Landsat
observations pulled from USGS, or the notebook stops with a clear error.

## Why this is a *template*, not just a one-off analysis

Everything specific to Atewa lives in **Section 1 only** — the boundary
polygon and the date list. Every other section is generic: search,
clip-first processing, trend analysis, visualization, and output
generation are all boundary-agnostic. To monitor a different reserve
(Bia, Kakum, Bosomtwe Range, Krokosua Hills — any of Ghana's 266+
forest reserves), a Forestry Commission analyst only needs to:

1. Replace the boundary polygon in Section 1 with the target reserve's
   coordinates (ideally the real Forestry Commission GIS boundary, not
   a representative approximation like the one used here)
2. Adjust the date range if monitoring a different period
3. Run the rest of the notebook unchanged

## Study area sourcing

Real area figure — **236.63 km²** — cross-validated independently by
two sources (IUCN's "23,663 ha" and a published tree checklist's
"236.63 km²" for the Atewa Range Reserve specifically, not including
the separate Atewa Extension Reserve). Shape constructed as a narrow
north-south corridor (matching the reserve's described "range of
hills" geometry) anchored to real, geocoded reference points: Kibi
(6.167°N, 0.550°W — the reserve lies on its western slopes), Anyinam
(6.382°N, 0.549°W — the range's northern end), and Asamankese
(5.860°N, 0.664°W — the range's southern end). As with the other
boundaries in this project, this is a representative construction from
verified published coordinates, not the official Forestry Commission
cadastral boundary — substitute that directly for operational use.


In [ ]:
from pathlib import Path
import numpy as np

from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery
from pygeofetch.models.download_task import DownloadOptions
from pygeofetch.processor import LandsatExtractor, TimeSeriesAnalyzer
from pygeofetch.processing.preprocessor import Preprocessor
from pygeofetch.viz import Plotter, MapViewer

print("Modules loaded: PyGeoFetch, LandsatExtractor, TimeSeriesAnalyzer, Preprocessor, Plotter, MapViewer")


## 1. Study Area — Atewa Range Forest Reserve (the only section specific to this reserve)


In [ ]:
import json

atewa_boundary_coords = [
    [-0.5579, 6.3821], [-0.5308, 6.3388], [-0.5417, 6.2739], [-0.5633, 6.2089],
    [-0.5904, 6.1440], [-0.6174, 6.0790], [-0.6337, 6.0141], [-0.6445, 5.9491],
    [-0.6553, 5.8842], [-0.6661, 5.8517], [-0.6120, 5.8679], [-0.6066, 5.9275],
    [-0.5958, 5.9924], [-0.5795, 6.0574], [-0.5579, 6.1223], [-0.5362, 6.1873],
    [-0.5200, 6.2522], [-0.5092, 6.3172], [-0.5146, 6.3713], [-0.5579, 6.3821],
]
atewa_geometry = {"type": "Polygon", "coordinates": [atewa_boundary_coords]}

atewa_lons = [c[0] for c in atewa_boundary_coords]
atewa_lats = [c[1] for c in atewa_boundary_coords]
aoi_extent = (min(atewa_lons), max(atewa_lons), min(atewa_lats), max(atewa_lats))

output_dir = Path("./data/atewa_deforestation_monitoring")
output_dir.mkdir(parents=True, exist_ok=True)

boundary_path = output_dir / "atewa_boundary.geojson"
boundary_path.write_text(json.dumps({
    "type": "FeatureCollection",
    "features": [{"type": "Feature", "properties": {"name": "Atewa Range Forest Reserve"}, "geometry": atewa_geometry}],
}))

from shapely.geometry import Polygon
import math
lat0 = sum(atewa_lats) / len(atewa_lats)
m_lat, m_lon = 111320, 111320 * math.cos(math.radians(lat0))
area_km2 = Polygon([(x*m_lon, y*m_lat) for x, y in atewa_boundary_coords]).area / 1e6

print("Study area: Atewa Range Forest Reserve, Eastern Region, Ghana")
print(f"Vertices: {len(atewa_boundary_coords)} (narrow N-S range, not a rectangle)")
print(f"Extent: {aoi_extent}")
print(f"Constructed area: {area_km2:.1f} km2 (real cross-validated figure: 236.63 km2)")

# ── TO REUSE THIS TEMPLATE FOR A DIFFERENT RESERVE ──
# Replace atewa_boundary_coords above with your reserve's real boundary,
# and update the search date range in Section 2 below. Nothing else in
# this notebook needs to change.


## 2. Search — multi-year Landsat time series (dry-season composites)

Same principle as the Obuasi mining-belt monitoring project: several
dates across multiple years give `TimeSeriesAnalyzer.trend()` a real
slope to fit, rather than a fragile two-date comparison. Dry-season
windows (Nov-Feb) keep cloud cover and vegetation phenology more
comparable across years.


In [ ]:
client = PyGeoFetch(log_level="INFO")

USGS_USERNAME = "your_ers_username"
USGS_APP_TOKEN = "your_application_token"
client.add_credentials("usgs", username=USGS_USERNAME, api_key=USGS_APP_TOKEN)

date_windows = [
    ("2018-11-01", "2019-02-28"), ("2019-11-01", "2020-02-28"),
    ("2020-11-01", "2021-02-28"), ("2021-11-01", "2022-02-28"),
    ("2022-11-01", "2023-02-28"), ("2023-11-01", "2024-02-28"),
]

scenes = {}
for start, end in date_windows:
    query = SearchQuery(
        geometry=atewa_geometry, start_date=start, end_date=end,
        satellites=["Landsat-8", "Landsat-9"], cloud_cover_max=20, max_results=5,
    )
    results = client.search(query, providers=["usgs"])
    results = [r for r in results if r.datetime is not None]
    if results:
        best = min(results, key=lambda r: r.cloud_cover or 100)
        scenes[str(best.datetime.date())] = best
        print(f"  {start[:4]}: {best.id}  ({best.datetime.date()}, {best.cloud_cover:.0f}% cloud)")
    else:
        print(f"  {start[:4]}: no usable scenes found")

if len(scenes) < 3:
    raise RuntimeError(
        f"Only {len(scenes)} usable date(s) found -- need at least 3 for a "
        f"meaningful trend. Widen the date windows or the cloud-cover "
        f"threshold and try again. No synthetic fallback is used."
    )
print(f"\n{len(scenes)} dates found -- proceeding with real data only.")


## 3. Download all dates


In [ ]:
download_results = {}
options = DownloadOptions(parallel=1, resume=True, on_failure="skip")

for date_str, scene in scenes.items():
    scene_dir = output_dir / date_str
    results_dl = client.download([scene], destination=scene_dir, options=options)
    r = results_dl[0]
    print(f"  {date_str}: {'OK' if r.success else f'FAILED: {r.error}'}")
    if r.success:
        download_results[date_str] = r

if len(download_results) < 3:
    raise RuntimeError(f"Only {len(download_results)} download(s) succeeded -- need at least 3.")


## 4. Clip-first processing: extract → clip → scale → cloud-mask

Every scaling, masking, and index computation operates only on the
actual reserve boundary, not the full downloaded scene.


In [ ]:
extractor = LandsatExtractor()
pp = Preprocessor()

# Atewa is a narrow, elongated N-S reserve (~59km tall, ~15km wide).
# Landsat's WRS-2 tiling means different acquisition dates can come from
# scenes with slightly different footprints over an AOI this shape --
# confirmed directly: one date's clipped RED band came back at 757x573
# pixels, another at 1959x573 -- identical width (same E-W extent) but
# very different height, consistent with one date's scene only partially
# covering the reserve's N-S extent while another covered it fully.
# TimeSeriesAnalyzer correctly refuses to stack mismatched grids rather
# than silently producing nonsense -- the fix is to snap every date onto
# one common reference grid (built once, from the AOI's own bounds at
# Landsat's native 30m resolution) before scaling or stacking, so a
# partial-coverage date correctly shows nodata outside its real coverage
# rather than being misaligned against the others.
import math
from rasterio.warp import transform_bounds, reproject, Resampling
from rasterio.transform import from_bounds as _from_bounds

def build_reference_grid(aoi_geometry, target_crs, resolution_m=30.0):
    lons = [c[0] for c in aoi_geometry["coordinates"][0]]
    lats = [c[1] for c in aoi_geometry["coordinates"][0]]
    bounds_utm = transform_bounds("EPSG:4326", target_crs, min(lons), min(lats), max(lons), max(lats))
    width = int(math.ceil((bounds_utm[2] - bounds_utm[0]) / resolution_m))
    height = int(math.ceil((bounds_utm[3] - bounds_utm[1]) / resolution_m))
    transform = _from_bounds(*bounds_utm, width, height)
    return transform, width, height


def snap_to_grid(src_path, dst_path, target_transform, target_width, target_height, target_crs):
    import rasterio
    with rasterio.open(src_path) as src:
        dst_arr = np.full((target_height, target_width), 0, dtype=src.dtypes[0])
        reproject(
            source=rasterio.band(src, 1), destination=dst_arr,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=target_transform, dst_crs=target_crs,
            resampling=Resampling.bilinear, dst_nodata=0,
        )
        profile = src.profile.copy()
        profile.update(height=target_height, width=target_width, transform=target_transform)
        with rasterio.open(dst_path, "w", **profile) as dst:
            dst.write(dst_arr, 1)
    return dst_path


def process_scene_clip_first(source, work_dir, aoi_geometry, bands=("red", "nir"), ref_grid=None):
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)

    individual = extractor._resolve_individual_files(source)
    if individual is not None:
        extracted = individual
        sensor_hint = next(iter(extracted.keys()), "")
    else:
        bundle_path = extractor._resolve_path(source)
        if bundle_path is None:
            return None
        extracted = extractor.extract_bundle(bundle_path, work_dir / "extracted")
        sensor_hint = bundle_path.name

    sensor = extractor._detect_sensor(sensor_hint)
    band_map = {"OLI": {"red": "SR_B4", "nir": "SR_B5"}, "TM": {"red": "SR_B3", "nir": "SR_B4"}}[sensor]

    clipped_paths = {}
    for band_name in list(bands) + ["qa_pixel"]:
        suffix = band_map.get(band_name) if band_name in band_map else "QA_PIXEL"
        raw_path = extractor.find_band(extracted, suffix)
        if raw_path is None:
            continue
        clip_result = pp.clip(raw_path, geometry=aoi_geometry, output=str(work_dir / f"{band_name}_clipped.tif"))
        if not clip_result.success:
            continue
        clipped_path = clip_result.output_path

        # Determine the reference grid once, from the first clipped
        # raster's real CRS -- every subsequent date snaps onto this
        # same grid regardless of its own scene's actual coverage.
        if ref_grid is None:
            import rasterio
            with rasterio.open(clipped_path) as src:
                ref_grid = build_reference_grid(aoi_geometry, src.crs)

        target_transform, target_width, target_height = ref_grid
        import rasterio
        with rasterio.open(clipped_path) as src:
            target_crs = src.crs
        snapped_path = work_dir / f"{band_name}_snapped.tif"
        snap_to_grid(clipped_path, snapped_path, target_transform, target_width, target_height, target_crs)
        clipped_paths[band_name] = snapped_path

    if not all(b in clipped_paths for b in bands):
        return None

    band_arrays, profile = {}, None
    for band_name in bands:
        arr, prof = extractor.load_scaled_band(clipped_paths[band_name])
        if profile is None:
            profile = prof
        band_arrays[band_name.upper()] = arr

    if "qa_pixel" in clipped_paths:
        mask = extractor.cloud_mask(clipped_paths["qa_pixel"], shape=next(iter(band_arrays.values())).shape)
        for k in band_arrays:
            band_arrays[k] = np.where(mask, np.nan, band_arrays[k])

    return band_arrays, profile, sensor, ref_grid


date_bands = {}
clip_profile = None
reference_grid = None
for date_str, result in download_results.items():
    processed = process_scene_clip_first(result, output_dir / date_str, atewa_geometry, ref_grid=reference_grid)
    if processed is None:
        print(f"  {date_str}: could not process (missing bands)")
        continue
    band_arrays, profile, sensor, reference_grid = processed
    clip_profile = clip_profile or profile
    date_bands[date_str] = {}

    import rasterio
    for band_key, arr in band_arrays.items():
        band_name = band_key
        out_path = output_dir / date_str / f"{band_name.lower()}_scaled_clipped.tif"
        prof = profile.copy()
        prof.update(dtype="float32", count=1)
        with rasterio.open(out_path, "w", **prof) as dst:
            dst.write(arr.astype(np.float32)[np.newaxis, :, :])
        date_bands[date_str][band_name] = out_path

    print(f"  {date_str}: clipped + snapped to common grid + scaled + masked ({sensor})")

if len(date_bands) < 3:
    raise RuntimeError(f"Only {len(date_bands)} date(s) processed successfully -- need at least 3.")


## 5. Build the NDVI time series and compute the real trend


In [ ]:
ts = TimeSeriesAnalyzer(index="NDVI")
stack = ts.build_index_stack(date_bands)
print(stack)

trend = ts.trend(stack)
print(f"\nTrend range: [{np.nanmin(trend):.4f}, {np.nanmax(trend):.4f}] NDVI/year")
print(f"Mean trend: {np.nanmean(trend):.4f} NDVI/year")

declining_pct = 100 * np.nanmean(trend < -0.02)
print(f"Area with clear declining trend (< -0.02 NDVI/year, likely deforestation): {declining_pct:.1f}%")


## 6. Zonal time series and anomaly detection


In [ ]:
zonal_df = ts.zonal_timeseries(stack, boundary_path, zone_id_field="name")
print(zonal_df.to_string(index=False))

baseline_dates = stack.dates[:2]
anomaly = ts.anomaly(stack, baseline=baseline_dates, target=stack.dates[-1])
print(f"\nAnomaly (z-score) range: [{np.nanmin(anomaly):.2f}, {np.nanmax(anomaly):.2f}]")
print(f"Area with strong negative anomaly (z < -2, likely recent clearance): {100*np.nanmean(anomaly < -2):.1f}%")


## 7. Visualization — strong, intuitive color communication

Environmental severity uses colors people already read intuitively:
deep red for severe loss, orange for moderate loss, green for stable
or recovering forest — not a generic diverging colormap that requires
a legend to interpret.


In [ ]:
pl = Plotter(figsize=(12, 9))

pl.plot_raster(
    trend, title="Atewa Forest Reserve — NDVI Trend",
    colormap="RdYlGn", vmin=-0.05, vmax=0.05, extent=aoi_extent,
    colorbar_label="NDVI / year (red = loss, green = gain)",
    output=str(output_dir / "atewa_trend_map.png"),
)


In [ ]:
DEFOR_LABELS = {0: "Severe loss", 1: "Moderate loss", 2: "Stable / intact forest", 3: "Recovering / gaining"}
DEFOR_COLORS = {0: "#a50026", 1: "#f46d43", 2: "#1a9850", 3: "#006837"}

def classify_trend(trend_arr):
    codes = np.full(trend_arr.shape, 2, dtype=np.int16)
    codes[trend_arr < -0.03] = 0
    codes[(trend_arr >= -0.03) & (trend_arr < -0.01)] = 1
    codes[trend_arr > 0.02] = 3
    codes[np.isnan(trend_arr)] = 2
    return codes

classified = classify_trend(trend)
pl.plot_classification(
    classified, class_labels=DEFOR_LABELS, class_colors=DEFOR_COLORS,
    title="Atewa Forest Reserve — Deforestation Severity Classification",
    extent=aoi_extent, output=str(output_dir / "atewa_classification.png"),
)

severe_pct = 100 * np.nanmean(classified == 0)
moderate_pct = 100 * np.nanmean(classified == 1)
print(f"Severe loss: {severe_pct:.1f}% | Moderate loss: {moderate_pct:.1f}% | Combined: {severe_pct+moderate_pct:.1f}%")
if severe_pct + moderate_pct > 5:
    print("\nRECOMMENDATION: this level of loss warrants ground verification")
    print("and formal reporting through Forestry Commission channels.")


## 8. Composite summary figure


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

im0 = axes[0].imshow(trend, cmap="RdYlGn", vmin=-0.05, vmax=0.05, extent=aoi_extent, aspect="auto")
axes[0].set_title("NDVI Trend", fontsize=13)
plt.colorbar(im0, ax=axes[0], fraction=0.04)

from matplotlib.colors import ListedColormap
cmap_class = ListedColormap([DEFOR_COLORS[i] for i in range(4)])
axes[1].imshow(classified, cmap=cmap_class, extent=aoi_extent, aspect="auto")
axes[1].set_title(f"Severity — {severe_pct+moderate_pct:.1f}% affected", fontsize=13)

zone_series = ts.zone_series(stack, boundary_path, zone_id="Atewa Range Forest Reserve", zone_id_field="name")
axes[2].plot(list(zone_series.keys()), list(zone_series.values()), marker="o", linewidth=2, color="#1a9850")
axes[2].set_title("Reserve-Wide Mean NDVI", fontsize=13)
axes[2].tick_params(axis="x", rotation=30)
axes[2].grid(alpha=0.3)

fig.suptitle("Atewa Range Forest Reserve — Deforestation Monitoring Summary", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(output_dir / "atewa_summary.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Interactive map


In [ ]:
mv = MapViewer(center=(6.11, -0.58), zoom=11)
mv.add_vector(str(boundary_path), layer_name="Atewa Range Forest Reserve",
              style={"color": "#1a1a1a", "weight": 2, "fillOpacity": 0.0})
from pygeofetch.processing.base import _safe_write_band
trend_tif = output_dir / "trend.tif"
_safe_write_band(trend, clip_profile, trend_tif)
mv.add_raster(str(trend_tif), colormap="RdYlGn", vmin=-0.05, vmax=0.05, layer_name="NDVI Trend", opacity=0.75)
mv.add_basemap("SATELLITE")
map_path = output_dir / "interactive_map.html"
mv.save(map_path)
print(f"Interactive map saved: {map_path}")


## 10. Save GIS-ready outputs


In [ ]:
results_dir = output_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
_safe_write_band(trend, clip_profile, results_dir / "ndvi_trend.tif")
_safe_write_band(classified.astype(np.float32), clip_profile, results_dir / "severity_classification.tif")
_safe_write_band(anomaly, clip_profile, results_dir / "anomaly.tif")
print(f"Saved to: {results_dir}")
for f in sorted(results_dir.glob("*.tif")):
    print(f"  {f.name}")


## 11. SAR corroboration — mitigating cloud cover, not just cross-checking

Atewa's upland evergreen forest sits in one of Ghana's cloudiest
climate zones. Optical time series in tropical forest regions are
routinely thin — many candidate scenes get filtered out by cloud cover
before they're even usable, which can leave real gaps in an NDVI-only
trend. This is not a hypothetical concern; it's the specific reason
operational tropical deforestation-alert systems (RADD, JICA-JAXA's
JJ-FAST) use Sentinel-1 radar rather than relying on optical data alone.

**The physical signal:** VH (cross-polarized) backscatter is
particularly sensitive to vegetation structure and above-ground
biomass — dense forest canopy produces strong volume scattering,
depolarizing the radar signal into VH. When canopy is cleared, that
volume-scattering structure is gone; VH backscatter drops sharply,
regardless of cloud cover, time of day, or season. This section reuses
the exact same clip-first SAR pipeline (`GRDExtractor`, CRS-aware
`clip()`, `calibrate()`, `despeckle()`) already built and verified for
the Accra flood-mapping project in this series — the same physics
(backscatter change detection) applied to a different problem.


In [ ]:
from pygeofetch.sar import SARProcessor, GRDExtractor

sar = SARProcessor()
grd_extractor_vh = GRDExtractor(polarisation="VH")  # VH, not VV -- canopy structure sensitivity

sar_windows = {
    "baseline":  ("2018-11-01", "2018-12-15"),
    "recent":    ("2023-11-01", "2023-12-15"),
}

sar_scenes = {}
for label, (start, end) in sar_windows.items():
    query = SearchQuery(
        geometry=atewa_geometry, start_date=start, end_date=end,
        product_type="GRD", polarisation="VH", max_results=5,
    )
    results = client.search(query, providers=["copernicus"])
    if results:
        best = results[0]
        sar_scenes[label] = best
        print(f"  {label}: {best.id} ({best.datetime})")
    else:
        print(f"  {label}: no scenes found")

have_sar = len(sar_scenes) == 2
if not have_sar:
    print("\nSAR corroboration skipped -- insufficient Sentinel-1 coverage found.")
    print("(This does not invalidate the optical NDVI trend above -- it just")
    print("means the cloud-independent cross-check below is unavailable this run.)")


In [ ]:
sar_calibrated = {}

if have_sar:
    sar_output_dir = output_dir / "sar"
    for label, scene in sar_scenes.items():
        date_dir = sar_output_dir / label
        results_dl = client.download([scene], destination=date_dir, options=options)
        if not results_dl[0].success:
            print(f"  {label}: download failed - {results_dl[0].error}")
            continue

        raw_vh = grd_extractor_vh.extract_band(results_dl[0], output_dir=date_dir, label=label)
        if raw_vh is None:
            continue
        clip_result = pp.clip(raw_vh, geometry=atewa_geometry, output=str(date_dir / f"{label}_vh_clipped.tif"))
        if not clip_result.success:
            print(f"  {label}: clip failed - {clip_result.error}")
            continue
        cal_result = sar.calibrate(str(clip_result.output_path), output_type="sigma0", in_db=True)
        despeckle_result = sar.despeckle(str(cal_result.output_path), filter="lee", window=5)
        sar_calibrated[label] = despeckle_result.output_path
        print(f"  {label}: extracted + clipped + calibrated + despeckled")

    have_sar = "baseline" in sar_calibrated and "recent" in sar_calibrated


In [ ]:
if have_sar:
    # Canopy loss = VH backscatter DROP, not a rise -- the mechanism
    # underlying flood_map()'s change detection is directly reusable
    # here (compare against a reference, flag pixels that dropped beyond
    # a threshold), just applied to a different physical phenomenon.
    # "decrease" is the correct direction for canopy loss specifically --
    # unlike Accra's flood mapping, there is no equivalent
    # "double-bounce increase" signature for forest clearing.
    canopy_result = sar.flood_map(
        str(sar_calibrated["recent"]), reference=str(sar_calibrated["baseline"]),
        threshold=-3.0, detect_direction="decrease",
    )
    print(f"SAR-detected canopy disturbance: {canopy_result.metadata['water_pct']:.1f}% of the reserve")

    import rasterio
    with rasterio.open(canopy_result.output_path) as src:
        sar_disturbance_mask = src.read(1)

    pl.plot_classification(
        sar_disturbance_mask,
        class_labels={0: "No significant VH change", 1: "Canopy disturbance (SAR)"},
        class_colors={0: "#e8e8e8", 1: "#8B0000"},
        title="Atewa — SAR-Detected Canopy Disturbance (VH backscatter drop)",
        extent=aoi_extent, output=str(output_dir / "atewa_sar_disturbance.png"),
    )


## 12. Combining optical and SAR — a higher-confidence disturbance map

Areas flagged by *both* the optical NDVI decline and the SAR VH
backscatter drop are a substantially higher-confidence deforestation
signal than either alone — this is standard practice in operational
tropical forest monitoring, precisely because each sensor has
different failure modes (optical: cloud gaps; SAR: occasional
speckle/terrain artifacts even after despeckling) that don't coincide.


In [ ]:
if have_sar:
    # Optical (Landsat, 30m native) and SAR (Sentinel-1 GRD, ~10m native)
    # have genuinely different resolutions -- not a bug, the real native
    # resolution difference between the two sensors. Aligned via
    # PyGeoFetch's own Preprocessor.resample(reference=...) -- not
    # hand-written rasterio code -- using "nearest" specifically, since
    # both rasters are categorical class values (0/1, 0-3), and bilinear
    # would blend adjacent classes into meaningless fractional values.
    # References the classification already written to disk in Section
    # 10, so this needs no separate write step of its own.
    align_result = pp.resample(
        canopy_result.output_path,
        reference=results_dir / "severity_classification.tif",
        method="nearest",
    )
    with rasterio.open(align_result.output_path) as f:
        aligned_sar_mask = f.read(1)

    optical_flag = classified <= 1  # severe or moderate loss
    combined = np.where(optical_flag & (aligned_sar_mask == 1), 2,
                np.where(optical_flag | (aligned_sar_mask == 1), 1, 0))

    pl.plot_classification(
        combined,
        class_labels={0: "No disturbance signal", 1: "Flagged by one sensor", 2: "Flagged by BOTH (high confidence)"},
        class_colors={0: "#f0f0f0", 1: "#fdae61", 2: "#8B0000"},
        title="Atewa — Combined Optical + SAR Disturbance Confidence",
        extent=aoi_extent, output=str(output_dir / "atewa_combined_confidence.png"),
    )
    high_confidence_pct = 100 * np.mean(combined == 2)
    print(f"High-confidence disturbance (flagged by BOTH optical and SAR): {high_confidence_pct:.1f}% of the reserve")


## Summary — for Forestry Commission adoption

**Optical + SAR, not optical alone.** Given Atewa's persistent tropical
cloud cover, this notebook combines Landsat NDVI trend analysis with a
Sentinel-1 VH backscatter cross-check — the same reasoning behind
operational systems like RADD and JJ-FAST. Where both signals agree,
confidence in a real disturbance is substantially higher than either
sensor alone could support.

This notebook is a template: to monitor a different reserve, replace
the boundary in Section 1 and the date list in Section 2. Everything
else — the clip-first pipeline, `TimeSeriesAnalyzer` trend/zonal/anomaly
computation, classification thresholds, and visualization — is reusable
without modification, and produces GIS-ready GeoTIFF outputs directly
importable into QGIS or ArcGIS for further work.

### References

- IUCN. Community-Led Governance and Management of Atewa Forest Range
  Landscape (area: 23,663 ha).
- Checklist of Trees in Atewa Range Forest Reserve, Ghana (area:
  236.63 km² for the core reserve).
- Forestry Commission of Ghana — reserve management authority.
